# GSM Thermodynamic Box - Interface-based Interactive Widget Demo

This notebook demonstrates the interface-based implementation of the GSM Thermodynamic Box with an interactive widget.

The GSMThermodynBox2 uses interface-based state function implementations (GSMStateFnIfc) instead of raw expressions, making the framework more modular and extensible.

In [2]:
import sympy as sp
from bmcs_matmod.gsm_lagrange.core2 import gsm_vars
from bmcs_matmod.gsm_lagrange.core2.gsm_state_fn import GSMStateFn, StateFunction
from bmcs_matmod.gsm_lagrange.core2.gsm_thermodyn_box import GSMThermodynBox
from bmcs_matmod.gsm_lagrange.core2.gsm_thermodyn_box_widget import create_interactive_widget

## Create Variables and Initial State Function

We'll create a simple thermodynamic state function to start with. Let's begin with the Helmholtz free energy F(T, ε, Ɛ) as our initial state function.

In [3]:
# Create symbolic variables using enhanced Scalar class
from bmcs_matmod.gsm_lagrange.core2.gsm_vars import Scalar

# Thermal variables
T = Scalar(r'\vartheta', codename='T', real=True, positive=True)
S = Scalar('S', codename='S', real=True)

# Mechanical variables  
eps = Scalar(r'\varepsilon', codename='eps', real=True)
sig = Scalar(r'\sigma', codename='sig', real=True)

# Damage variables
omega = Scalar(r'\omega', codename='omega', real=True, positive=True)
Y = Scalar('Y', codename='Y', real=True)

# Material parameters
c_T = Scalar(r'c_T', codename='c_T', positive=True)
E = Scalar('E', codename='E', positive=True)
H = Scalar('H', codename='H', positive=True)

# Create enhanced Helmholtz free energy expression F(T, ε, ω) to match roundtrip notebook
# Additional thermal parameters
C_eps = Scalar(r'C_{\varepsilon}', codename='C_eps', real=True, nonnegative=True)
T_0 = Scalar(r'\vartheta_0', codename='T_0', real=True, nonnegative=True)

# Extended Helmholtz free energy: elastic-damage + thermal capacity
F_elastic = sp.Rational(1, 2) * (1 - omega) * E * eps**2
F_thermal = C_eps * (T - T * sp.log(T / T_0))
F_expr = F_elastic + F_thermal

## Create State Function Instance and Thermodynamic Box

Now we create a GSMStateFn instance for the Helmholtz free energy and initialize the GSMThermodynBox2.

In [4]:
# Create the initial state function instance (Helmholtz free energy)
F_state_fn = GSMStateFn(
    fn_expr=F_expr,
    th_x_var=T,      # Temperature is natural variable
    th_y_var=S,      # Entropy is conjugate variable
    mc_x_var=eps,    # Strain is natural variable
    mc_y_var=sig,    # Stress is conjugate variable
    Eps_var=omega,   # Internal natural variable (damage)
    Sig_var=Y,       # Internal conjugate variable (stored energy)
    state_function_type=StateFunction.HELMHOLTZ
)

# Create the thermodynamic box
gsm_box = GSMThermodynBox(
    initial_state_fn=StateFunction.HELMHOLTZ,
    initial_state_instance=F_state_fn,
)

## Create and Display Interactive Widget

Now let's create the interactive widget to explore the thermodynamic relationships.

In [ ]:
# Create the interactive widget
widget = create_interactive_widget(
    gsm_box=gsm_box,
    title="GSM Thermodynamic Box 2 - Interface-based Interactive Widget"
)

# Display the widget
widget.show()

In [6]:
import sympy as sp

# Define a symbol for the variable
x = sp.Symbol('x', real=True)

# Define the expression log(exp(x))
# In SymPy, sp.log is the natural logarithm (ln), and sp.exp is the exponential function
expr = sp.log(sp.exp(x))

print("Original expression:", expr)
print("Type of expression:", type(expr))

# Simplify the expression
simplified = sp.simplify(expr)
print("Simplified expression:", simplified)

# Test with our GSMStateFn simplification method
simplified_with_timeout = GSMStateFn._simplify_with_timeout(expr)
print("Simplified with timeout method:", simplified_with_timeout)

# More complex examples
print("\n--- More Examples ---")

# log(exp(2*x))
expr2 = sp.log(sp.exp(2*x))
print(f"log(exp(2*x)) = {expr2} → simplified: {sp.simplify(expr2)}")

# log(exp(x + 1))
expr3 = sp.log(sp.exp(x + 1))
print(f"log(exp(x + 1)) = {expr3} → simplified: {sp.simplify(expr3)}")

# log(exp(x)*exp(y))
y = sp.Symbol('y', real=True)
expr4 = sp.log(sp.exp(x) * sp.exp(y))
print(f"log(exp(x)*exp(y)) = {expr4} → simplified: {sp.simplify(expr4)}")

# For comparison, if you need log base 10, use sp.log(expr, 10)
expr5 = sp.log(sp.exp(x), 10)  # log_10(e^x)
print(f"log_10(exp(x)) = {expr5} → simplified: {sp.simplify(expr5)}")

# Show that SymPy recognizes these as inverses
print(f"\nSymPy knows that log and exp are inverses:")
print(f"log(exp(x)) should equal x: {sp.simplify(sp.log(sp.exp(x))) == x}")
print(f"exp(log(x)) should equal x (for x > 0): {sp.simplify(sp.exp(sp.log(x)))}")  # Note: assumes x > 0

Original expression: x
Type of expression: <class 'sympy.core.symbol.Symbol'>
Simplified expression: x
Simplified with timeout method: x

--- More Examples ---
log(exp(2*x)) = 2*x → simplified: 2*x
log(exp(x + 1)) = x + 1 → simplified: x + 1
log(exp(x)*exp(y)) = log(exp(x)*exp(y)) → simplified: x + y
log_10(exp(x)) = x/log(10) → simplified: x/log(10)

SymPy knows that log and exp are inverses:
log(exp(x)) should equal x: True
exp(log(x)) should equal x (for x > 0): x
